#DAY 12 Databricks Challenge

##Challenges
### 🛠️ Tasks:

1. Train simple regression model
2. Log parameters, metrics, model
3. View in MLflow UI
4. Compare runs

####OPTIONAL ONLY IF GOLD TABLE DON'T EXIST

In [0]:
#Creating Gold Aggregation
from pyspark.sql import functions as F

silver = spark.table("workspace.default.silver_events_part")

gold_products = silver.groupBy("product_id").agg(
    F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
    F.sum(
        F.when(F.col("event_type") == "purchase", F.col("price"))
         .otherwise(0)
    ).alias("revenue"),
    F.count("*").alias("total_events")
)


In [0]:
#Adding conversion rates
gold_products = gold_products.withColumn(
    "conversion_rate",
    F.when(F.col("views") > 0,
           F.col("purchases") / F.col("views"))
     .otherwise(0)
)


In [0]:
#Verifying before writing into table
gold_products.show(10)
gold_products.printSchema()


In [0]:
#Writing to delta table
gold_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.products")


In [0]:
#Verifying the data from Gold table
spark.table("gold.products").show(5)


####Task 1 - Train simple regression model

#####Step 1: Preparing the training data

In [0]:
#Converts Spark DataFrame → Pandas DataFrame(Required for Scikit-learn)
df = spark.table("gold.products").toPandas()
display(len(df),df)


In [0]:
#Select features and label
df_ml = df[["views", "purchases"]]
display(len(df_ml),df_ml)

In [0]:
#Cleaning the data without na values
df_ml = df_ml.dropna()
print("Rows after cleaning:", len(df_ml))


In [0]:
#Splitting Training and Test Data
from sklearn.model_selection import train_test_split

X = df_ml[["views"]]        # feature
y = df_ml["purchases"]     # label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [0]:
#Training Linear Regression Model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Coefficient:", model.coef_)
print("Intercept:", model.intercept_)


#**************************************************

####Task 2 - Log Parameters, metrics and model

In [0]:
#Starting an MLflow run
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="linear_regression_v1"):

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("feature_used", "views")
    mlflow.log_param("test_size", 0.2)

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluate
    r2_score = model.score(X_test, y_test)
    mlflow.log_metric("r2_score", r2_score)

    # Log model
    mlflow.sklearn.log_model(model, "model")

print(f"R² Score: {r2_score:.4f}")


#**************************************************

####Task 3 - Views in MLflow UI

####OPEN Experiments (Left pane) -----> Find the recently created experiment.

we will see:
- Parameters → model_type, test_size
- Metrics → r2_score
- Artifacts → model

#**************************************************

####Task 4 - Compare runs

In [0]:
#Select features and label
df_ml = df[["views", "purchases","revenue"]]
display(len(df_ml),df_ml)


In [0]:
#Cleaning the data without na values
df_ml = df_ml.dropna()
print("Rows after cleaning:", len(df_ml))


In [0]:
#Running a second experiement by taking two features for comparison
X = df_ml[["views", "revenue"]]
y = df_ml["purchases"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

with mlflow.start_run(run_name="linear_regression_v2"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features_used", "views,revenue")
    mlflow.log_param("test_size", 0.2)

    model = LinearRegression()
    model.fit(X_train, y_train)

    r2_score = model.score(X_test, y_test)
    mlflow.log_metric("r2_score", r2_score)

    mlflow.sklearn.log_model(model, "model")

print(f"R² Score (v2): {r2_score:.4f}")


#####Compare in MLflow UI between V1 and V2

- Go to Experiments
- Select both runs
- Click Compare

You can now compare:

- R² scores
- Parameters
- Feature sets

#**************************************************


%md
## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 